In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from data.clean_data import basic_clean
from data.load_data import load_raw_data
from eda.summary import data_summary
from data.feature_engineering import prepare_model_data

In [ ]:
df_raw = load_raw_data()
df = basic_clean(df_raw)

target = "annual_medical_cost"
log_target = "log1p_annual_medical_cost"

df = prepare_model_data(
    df,
    target_col=target,
    add_log=True,
    add_groups=True,
    add_tail_flags=False
)

print("Shape:", df.shape)
display(df.head())
display(data_summary(df))

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df[target].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.995]))
print("\nSkewness:", df[target].skew())
print("Kurtosis:", df[target].kurt())

mean_cost = df[target].mean()
median_cost = df[target].median()
p99_cost = df[target].quantile(0.99)

mean_log_cost = df[log_target].mean()
median_log_cost = df[log_target].median()

In [ ]:
plt.figure(figsize=(10, 6), dpi=150)

sns.histplot(
    data=df,
    x=target,
    bins=60,
    alpha=0.85,
    edgecolor="white"
)

plt.axvline(
    mean_cost,
    linestyle="--",
    linewidth=2,
    label="Mean"
)

plt.axvline(
    median_cost,
    linestyle=":",
    linewidth=2,
    label="Median"
)

plt.title("Distribution of Annual Medical Cost")
plt.xlabel("Annual Medical Cost")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_body = df[df[target] <= p99_cost]

plt.figure(figsize=(10, 6), dpi=150)

sns.histplot(
    data=df_body,
    x=target,
    bins=60,
    alpha=0.85,
    edgecolor="white"
)

plt.axvline(
    df_body[target].mean(),
    linestyle="--",
    linewidth=2,
    label="Mean"
)

plt.axvline(
    df_body[target].median(),
    linestyle=":",
    linewidth=2,
    label="Median"
)

plt.title("Distribution of Annual Medical Cost (Up to 99th Percentile)")
plt.xlabel("Annual Medical Cost")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6), dpi=150)

sns.histplot(
    data=df,
    x=log_target,
    bins=60,
    alpha=0.85,
    edgecolor="white"
)

plt.axvline(
    mean_log_cost,
    linestyle="--",
    linewidth=2,
    label="Mean"
)

plt.axvline(
    median_log_cost,
    linestyle=":",
    linewidth=2,
    label="Median"
)

plt.title("Distribution of log(1 + Annual Medical Cost)")
plt.xlabel("log(1 + Annual Medical Cost)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 3), dpi=150)

sns.boxplot(
    data=df,
    x=log_target,
    showfliers=True
)

plt.title("Boxplot of log(1 + Annual Medical Cost)")
plt.xlabel("log(1 + Annual Medical Cost)")
plt.tight_layout()
plt.show()

In [ ]:
df_plot = df.sample(5000, random_state=42).copy()

rng = np.random.default_rng(42)
df_plot["age_jitter"] = df_plot["age"] + rng.uniform(
    -0.25,
    0.25,
    size=len(df_plot)
)

plt.figure(figsize=(10, 6), dpi=150)

sns.scatterplot(
    data=df_plot,
    x="age_jitter",
    y=log_target,
    hue="smoker",
    alpha=0.35,
    s=25,
    edgecolor=None
)

plt.title("log(1 + Annual Medical Cost) vs Age")
plt.xlabel("Age")
plt.ylabel("log(1 + Annual Medical Cost)")
plt.legend(title="Smoker")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6), dpi=150)

sns.boxplot(
    data=df,
    x="hospitalizations_last_3yrs",
    y=log_target,
    showfliers=True
)

plt.title("log(1 + Annual Medical Cost) by Hospitalisations in Last 3 Years")
plt.xlabel("Hospitalisations in Last 3 Years")
plt.ylabel("log(1 + Annual Medical Cost)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6), dpi=150)

sns.boxplot(
    data=df,
    x="chronic_count",
    y=log_target,
    showfliers=True
)

plt.title("log(1 + Annual Medical Cost) by Chronic Count")
plt.xlabel("Chronic Count")
plt.ylabel("log(1 + Annual Medical Cost)")
plt.tight_layout()
plt.show()

In [ ]:
print(df["hospitalizations_last_3yrs"].value_counts().sort_index())
print()
print(df["chronic_count"].value_counts().sort_index())

In [ ]:
print(df["hospitalizations_grouped"].value_counts().sort_index())
print()
print(df["chronic_count_grouped"].value_counts().sort_index())

In [ ]:
hospital_grouped_summary = (
    df.groupby("hospitalizations_grouped", observed=True)["annual_medical_cost"]
    .agg(
        n="size",
        mean="mean",
        median="median",
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
    )
    .round(2)
)

hospital_grouped_summary

In [ ]:
chronic_grouped_summary = (
    df.groupby("chronic_count_grouped", observed=True)["annual_medical_cost"]
    .agg(
        n="size",
        mean="mean",
        median="median",
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
    )
    .round(2)
)

chronic_grouped_summary

In [ ]:
hospital_order = ["0", "1", "2+"]

plt.figure(figsize=(10, 6), dpi=150)

sns.boxplot(
    data=df,
    x="hospitalizations_grouped",
    y=log_target,
    order=hospital_order,
    showfliers=True
)

plt.title("log(1 + Annual Medical Cost) by Grouped Hospitalisations")
plt.xlabel("Hospitalisations in Last 3 Years")
plt.ylabel("log(1 + Annual Medical Cost)")
plt.tight_layout()
plt.show()

In [ ]:
chronic_order = ["0", "1", "2", "3", "4+"]

plt.figure(figsize=(10, 6), dpi=150)

sns.boxplot(
    data=df,
    x="chronic_count_grouped",
    y=log_target,
    order=chronic_order,
    showfliers=True
)

plt.title("log(1 + Annual Medical Cost) by Grouped Chronic Count")
plt.xlabel("Chronic Count")
plt.ylabel("log(1 + Annual Medical Cost)")
plt.tight_layout()
plt.show()